In [1]:
import yt_dlp

In [2]:
url = "https://www.youtube.com/watch?v=H26xnpL-ei0"

In [3]:
from pydub import AudioSegment
AudioSegment.converter = r"C:\ffmpeg\bin\ffmpeg.exe"  

c:\Users\playdata2\miniconda3\envs\env1\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [4]:
setting = {
    'format': 'bestvideo+bestaudio/best',
    'outtmpl': '%(title)s.%(ext)s',
    'keepvideo': True,
    'ffmpeg_location': r'C:\ffmpeg\bin\ffmpeg.exe',  # 여기 경로 바꿔
    'postprocessors': [
        {
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }
    ],
    'restrictfilenames': True
}

In [5]:
with yt_dlp.YoutubeDL(setting) as ydl:
    ydl.download(url)

[youtube] Extracting URL: https://www.youtube.com/watch?v=H26xnpL-ei0
[youtube] H26xnpL-ei0: Downloading webpage


[youtube] H26xnpL-ei0: Downloading android vr player API JSON
[info] H26xnpL-ei0: Downloading 1 format(s): 399+251
[download] NVIDIA_GTC_2026_Open_Models_Panel_Highlights_with_Jensen_Huang.webm has already been downloaded
[ExtractAudio] Destination: NVIDIA_GTC_2026_Open_Models_Panel_Highlights_with_Jensen_Huang.mp3


In [6]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()


client = OpenAI()


In [9]:
with open(r"C:\Users\playdata2\OneDrive\Desktop\20260312_openapikey\NVIDIA_GTC_2026_Open_Models_Panel_Highlights_with_Jensen_Huang.mp3", 'rb') as f:
    transcript = client.audio.transcriptions.create(
        model='whisper-1',
        response_format='srt',
        file=f
    )


APIStatusError: Error code: 413 - {'error': {'message': '413: Maximum content size limit (26214400) exceeded (26215798 bytes read)', 'type': 'server_error', 'param': None, 'code': None}}

In [ ]:
"C:\Users\playdata2\OneDrive\Desktop\NVIDIA_GTC_2026_Open_Models_Panel_Highlights_with_Jensen_Huang.mp3"

In [13]:
import os
from openai import OpenAI
from pydub import AudioSegment

client = OpenAI() # 환경변수에 API 키가 세팅되어 있다고 가정합니다.

# 1. 원본 파일 경로
file_path = r"C:\Users\playdata2\OneDrive\Desktop\NVIDIA_GTC_2026_Open_Models_Panel_Highlights_with_Jensen_Huang.mp3"
print("오디오 파일을 불러오는 중입니다 (시간이 조금 걸릴 수 있습니다)...")
audio = AudioSegment.from_mp3(file_path)

# 2. 10분 단위(600,000 밀리초)로 오디오 쪼개기
# (10분 정도면 25MB 제한에 절대 걸리지 않습니다)
chunk_length_ms = 10 * 60 * 1000 
chunks = [audio[i:i + chunk_length_ms] for i in range(0, len(audio), chunk_length_ms)]

full_transcript = ""

# 3. 쪼개진 파일들을 하나씩 API에 보내서 텍스트 받아오기
for i, chunk in enumerate(chunks):
    chunk_name = f"temp_chunk_{i}.mp3"
    
    # 임시 파일로 저장
    chunk.export(chunk_name, format="mp3")
    print(f"[{i+1}/{len(chunks)}] 번째 조각 변환 중...")
    
    # API 요청
    with open(chunk_name, 'rb') as f:
        transcript = client.audio.transcriptions.create(
            model='whisper-1',
            response_format='srt', # 자막 형태
            file=f
        )
        full_transcript += transcript + "\n\n"
        
    # 다 쓴 임시 파일은 삭제
    os.remove(chunk_name)

print("\n--- 전체 변환 완료! ---")
# 결과물을 txt 파일로 저장하거나 출력할 수 있습니다.
print(full_transcript)

오디오 파일을 불러오는 중입니다 (시간이 조금 걸릴 수 있습니다)...


FileNotFoundError: [WinError 2] 지정된 파일을 찾을 수 없습니다